# Section 10: AI Agents — Foundations, Architectures & Protocols — Hands-On

### GenAI Decoded by Nij — Building Agents from First Principles

---

In **Sections 1–9**, you built everything an LLM needs to be useful: prompting, RAG, function calling, frameworks, fine-tuning, and deployment. But in every case, **you** decided the steps — the model just executed them.

An **agent** flips that. You give it a *goal*, and it figures out the steps: which tools to call, in what order, how to handle failures, and when to stop.

This notebook builds agents from scratch — no frameworks, just Python + the OpenAI API. You'll:

1. **Build the agent loop** — the `for` loop that makes LLMs autonomous
2. **Implement three architectures** — ReAct, Plan-and-Execute, Reflexion
3. **Build an MCP server** — expose tools via the Model Context Protocol
4. **Connect MCP to your agent** — one server, any client
5. **Compare architectures** — same task, different approaches

By the end, you'll understand what every agent framework (LangGraph, CrewAI, OpenAI Agents SDK) is doing under the hood.


## Setup

**Python 3.10+ recommended.** We'll use:

- **OpenAI** (`gpt-4o-mini`) — main model, matches Sections 1–9
- **Groq** (`llama-3.1-8b-instant`) — fast model for Plan-and-Execute executor
- **mcp** — Model Context Protocol SDK for building tool servers
- **python-dotenv** — environment variable loading

Make sure your `.env` has `OPENAI_API_KEY` and `GROQ_API_KEY`.


In [ ]:
import sys, subprocess, importlib.util

pkgs = {"openai": "openai", "groq": "groq", "python-dotenv": "dotenv"}
missing = [p for p, m in pkgs.items() if importlib.util.find_spec(m) is None]
if missing:
    print(f"📦 Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
print(f"✅ All packages ready — Python {sys.version.split()[0]}")


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

for key in ["OPENAI_API_KEY", "GROQ_API_KEY"]:
    status = "✅" if os.getenv(key) else "❌ MISSING"
    print(f"{status}  {key}")


In [ ]:
from openai import OpenAI
from groq import Groq
import json, time

oai = OpenAI()
groq_client = Groq()

OPENAI_MODEL = "gpt-4o-mini"
GROQ_MODEL = "llama-3.1-8b-instant"

def call_llm(prompt, model=OPENAI_MODEL, system=None, tools=None, messages=None):
    """Unified LLM call helper."""
    if messages is None:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
    client = groq_client if "llama" in model else oai
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
    return client.chat.completions.create(**kwargs).choices[0].message

print(f"✅ OpenAI → {OPENAI_MODEL}")
print(f"✅ Groq   → {GROQ_MODEL}")


---

## 3. The Agent Loop

This is the single most important pattern in this notebook. Every agent — whether built with LangGraph, CrewAI, or raw Python — runs the same core loop:

1. **Send** the current conversation (including tool results) to the LLM
2. **Check** if the LLM wants to call a tool or give a final answer
3. If **tool call** → execute it, append the result, go back to step 1
4. If **final answer** → return it, exit the loop

The LLM decides when to stop. Your code decides the ceiling.


### 3.1 — Define Tools and Schemas


In [ ]:
# ── Three tools our agent can use ──

def search_web(query: str) -> str:
    """Simulate a web search with realistic results."""
    knowledge = {
        "bitcoin price": "Bitcoin is currently trading at $68,420 (June 2026), up 4.2% this week.",
        "abu dhabi population": "Abu Dhabi has a population of approximately 1.57 million (2025 est).",
        "tesla revenue": "Tesla reported $25.7B revenue in Q4 2025, with 8.2% operating margin.",
        "python 3.13": "Python 3.13 released Oct 2024 with JIT compiler and free-threading.",
    }
    for key, val in knowledge.items():
        if key in query.lower():
            return val
    return f"Search for '{query}': No specific data found. Try more specific terms."

def calculate(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: invalid characters. Use only numbers and +-*/.()"
        return f"{expression} = {eval(expression)}"
    except Exception as e:
        return f"Error: {e}"

def get_current_date() -> str:
    """Return the current date."""
    from datetime import datetime
    return datetime.now().strftime("Today is %A, %B %d, %Y. Time: %H:%M")

# Tool schemas for the LLM
TOOLS = [
    {"type": "function", "function": {
        "name": "search_web",
        "description": "Search the web for current information about events, prices, stats. Returns text summary.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Search query — be specific"}
        }, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate a math expression. Examples: '100 * 0.15', '2**10', '(500 - 350) / 500 * 100'",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Math expression to evaluate"}
        }, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "get_current_date",
        "description": "Get today's date and current time.",
        "parameters": {"type": "object", "properties": {}}
    }},
]

TOOL_FUNCTIONS = {"search_web": search_web, "calculate": calculate, "get_current_date": get_current_date}
print(f"✅ {len(TOOLS)} tools defined: {list(TOOL_FUNCTIONS.keys())}")
print(f"   Test: {search_web('bitcoin price')}")
print(f"   Test: {calculate('2**10 + 3**5')}")


### 3.2 — The Agent Loop

Here it is — the core pattern. Every agent framework wraps these ~35 lines.


In [ ]:
def agent(user_question, tools=TOOLS, tool_functions=TOOL_FUNCTIONS, max_iterations=10):
    """A complete agent — from scratch, no framework."""
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. "
            "Think step by step. Use tools when you need information you don't have. "
            "When you have enough information, respond directly to the user. "
            "For common knowledge questions, answer directly without tools."},
        {"role": "user", "content": user_question}
    ]

    print(f"🧑 USER: {user_question}")
    print("─" * 60)

    for i in range(max_iterations):
        response = oai.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=tools,
        )
        msg = response.choices[0].message
        messages.append(msg)

        # ── Decision point: tool call or final answer? ──
        if msg.tool_calls:
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)

                # Safety: validate tool name before execution
                if fn_name not in tool_functions:
                    result = f"Error: tool '{fn_name}' not found. Available: {list(tool_functions.keys())}"
                else:
                    result = tool_functions[fn_name](**fn_args)

                print(f"  🔧 [{i+1}] {fn_name}({fn_args}) → {result[:100]}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            # No tool call = agent decided it's done
            print("─" * 60)
            print(f"🤖 ANSWER ({i+1} iterations): {msg.content}")
            return msg.content

    print(f"⚠️  Max iterations ({max_iterations}) reached")
    return "Could not complete within iteration limit."

print("✅ Agent function defined")


### 3.3 — Test the Agent

Watch how the agent makes different decisions for each query.


In [ ]:
# Needs search
agent("What's the current price of Bitcoin and how has it moved this week?")


In [ ]:
# Needs calculation
agent("What's a 15% tip on $87.50?")


In [ ]:
# Needs multiple tools (search + calculate)
agent("What's Tesla's Q4 2025 revenue, and what's that per month?")


In [ ]:
# No tools needed — should answer directly
agent("What is the capital of France?")


💡 **Key takeaway:** The agent made different decisions for each query — sometimes one tool, sometimes two in sequence, sometimes none. The `for` loop + `if msg.tool_calls` branch is the entire mechanic. Every agent framework wraps this same pattern with extra features (state, persistence, routing) — but this is the core.


---

## 4. ReAct — Explicit Reasoning Traces

The agent above *is* a ReAct agent — we just didn't make the reasoning visible. Let's add explicit Thought/Action/Observation logging so every decision is traceable.


In [ ]:
def react_agent(question, tools=TOOLS, tool_functions=TOOL_FUNCTIONS, max_iter=10):
    """ReAct agent with explicit Thought/Action/Observation traces."""
    messages = [
        {"role": "system", "content":
            "You are a research assistant using the ReAct framework. "
            "For EVERY step: 1) Write 'Thought:' explaining your reasoning, "
            "2) Call a tool if needed, 3) After seeing results think again, "
            "4) When you have enough info, give your final answer."},
        {"role": "user", "content": question}
    ]

    print(f"🧑 QUESTION: {question}")
    print("═" * 60)

    for i in range(max_iter):
        response = oai.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        messages.append(msg)

        if msg.content:
            print(f"\n💭 THOUGHT ({i+1}): {msg.content[:300]}")

        if msg.tool_calls:
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                result = tool_functions.get(fn_name, lambda **k: "Unknown tool")(**fn_args)
                print(f"⚡ ACTION: {fn_name}({json.dumps(fn_args)})")
                print(f"👁 OBSERVE: {result[:200]}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            print(f"\n{'═' * 60}")
            print(f"✅ FINAL ANSWER ({i+1} steps):\n{msg.content}")
            return msg.content

    return "Max iterations reached"

print("✅ ReAct agent defined")


In [ ]:
react_agent("What's Abu Dhabi's population, and what's the population density if the city covers 972 sq km?")


💡 **Key takeaway:** Making reasoning explicit gives you a debuggable trace. When something goes wrong, read the `Thought` step to see *why* the agent chose that action. ReAct's biggest advantage isn't the architecture — it's the debugging strategy.


---

## 5. Plan-and-Execute

ReAct is reactive — one step at a time. Plan-and-Execute is deliberate: a **planner** creates the full roadmap first, then an **executor** follows it step by step. Key trick: use a *stronger model* for planning, *cheaper model* for execution.


In [ ]:
def plan_and_execute(question, tools=TOOLS, tool_functions=TOOL_FUNCTIONS):
    """Plan-and-Execute: planner creates roadmap, executor follows it."""

    # Phase 1: PLAN
    plan_msg = call_llm(
        prompt=f"Break this task into 2-5 concrete steps. Return ONLY a JSON array.\n\nTask: {question}",
        system="Return only a JSON array like [\"step1\", \"step2\"]. No other text."
    )
    try:
        plan_text = plan_msg.content.strip()
        if plan_text.startswith("```"):
            plan_text = plan_text.split("\n", 1)[1].rsplit("```", 1)[0]
        steps = json.loads(plan_text)
    except:
        print("⚠️ Plan parsing failed, falling back to ReAct")
        return react_agent(question, tools, tool_functions)

    print(f"🧑 QUESTION: {question}")
    print("═" * 60)
    print(f"📋 PLAN ({len(steps)} steps):")
    for i, step in enumerate(steps):
        print(f"   {i+1}. {step}")
    print("─" * 60)

    # Phase 2: EXECUTE each step
    context = []
    for i, step in enumerate(steps):
        print(f"\n▶ STEP {i+1}: {step}")
        ctx_text = "\n".join(context) if context else "No previous context."

        resp = oai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "Execute this step. Use tools if needed. Be concise."},
                {"role": "user", "content": f"Previous results:\n{ctx_text}\n\nCurrent step: {step}"}
            ], tools=tools)
        exec_msg = resp.choices[0].message

        step_msgs = [
            {"role": "system", "content": "Complete this step."},
            {"role": "user", "content": f"Previous:\n{ctx_text}\n\nStep: {step}"},
            exec_msg]

        if exec_msg.tool_calls:
            for tc in exec_msg.tool_calls:
                fn = tc.function.name
                args = json.loads(tc.function.arguments)
                result = tool_functions.get(fn, lambda **k: "Unknown tool")(**args)
                print(f"   🔧 {fn}({args}) → {result[:120]}")
                step_msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
            followup = oai.chat.completions.create(model=OPENAI_MODEL, messages=step_msgs)
            step_result = followup.choices[0].message.content
        else:
            step_result = exec_msg.content

        print(f"   ✓ {step_result[:150]}")
        context.append(f"Step {i+1}: {step_result}")

    # Phase 3: SYNTHESIZE
    print(f"\n{'═' * 60}")
    final = call_llm(
        prompt=f"Question: {question}\n\nResults:\n" + "\n".join(context) + "\n\nSynthesize a clear answer.",
        system="Combine step results into one well-written answer.")
    print(f"✅ FINAL ANSWER:\n{final.content}")
    return final.content

print("✅ Plan-and-Execute agent defined")


In [ ]:
plan_and_execute("What was Tesla's Q4 2025 revenue? Calculate the monthly revenue, and tell me today's date.")


💡 **Key takeaway:** Plan-and-Execute separates *thinking* from *doing*. The plan is created once, each step executes independently. Best for tasks with 4+ predictable steps — reports, research, data analysis.


---

## 6. Reflexion — Self-Critique Loops

Reflexion adds self-evaluation: the agent produces output, checks if it's correct, reflects on failures, and retries. Most powerful for **verifiable output** — code, math, SQL — where you can test the result.


In [ ]:
def reflexion_agent(task, evaluator, max_attempts=3):
    """Reflexion: try → evaluate → reflect → retry."""
    reflections = []
    solution = None

    for attempt in range(1, max_attempts + 1):
        prompt = f"Task: {task}"
        if reflections:
            prompt += "\n\nPrevious reflections:"
            for r in reflections:
                prompt += f"\n- Attempt {r['attempt']}: {r['reflection']}"
            prompt += "\nUse these to improve your solution."

        solution = call_llm(
            prompt=prompt,
            system="You are a Python expert. Return ONLY the code. No markdown fences, no explanation."
        ).content

        print(f"\n{'═' * 60}")
        print(f"🔄 ATTEMPT {attempt}")
        print(f"📝 Solution:\n{solution[:300]}")

        result = evaluator(task, solution)
        status = "✅ PASS" if result["passed"] else "❌ FAIL"
        print(f"\n📊 {status} (Score: {result['score']}/100)")
        print(f"   {result['feedback']}")

        if result["passed"]:
            print(f"\n🎉 Solution accepted on attempt {attempt}!")
            return solution

        reflection = call_llm(
            prompt=f"My solution:\n{solution}\n\nFeedback:\n{result['feedback']}\n\nWhat went wrong? Be specific.",
            system="You are a code reviewer. Identify the exact bug and how to fix it."
        ).content

        reflections.append({"attempt": attempt, "reflection": reflection})
        print(f"\n🪞 Reflection: {reflection[:200]}")

    return solution

print("✅ Reflexion agent defined")


In [ ]:
def code_evaluator(task, solution):
    """Evaluate Python code against test cases."""
    code_text = solution.strip()
    if code_text.startswith("```"):
        code_text = code_text.split("\n", 1)[1].rsplit("```", 1)[0]

    tests = [
        (([3,1,4,1,5,9,2,6], 3), [1,1,2]),
        (([10,20,30], 2), [10,20]),
        (([], 5), []),
        (([7], 0), []),
        (([5,3,8,1,9], 5), [1,3,5,8,9]),
    ]

    try:
        ns = {}
        exec(code_text, ns)
        func = next((v for k, v in ns.items() if callable(v) and k != "__builtins__"), None)
        if not func:
            return {"passed": False, "score": 0, "feedback": "No function found in solution"}

        passed, failures = 0, []
        for args, expected in tests:
            try:
                r = func(*args)
                if r == expected:
                    passed += 1
                else:
                    failures.append(f"  {func.__name__}{args} → {r}, expected {expected}")
            except Exception as e:
                failures.append(f"  {func.__name__}{args} → ERROR: {e}")

        score = int(passed / len(tests) * 100)
        fb = f"{passed}/{len(tests)} tests passed."
        if failures:
            fb += " Failures:\n" + "\n".join(failures)
        return {"passed": passed == len(tests), "score": score, "feedback": fb}

    except SyntaxError as e:
        return {"passed": False, "score": 0, "feedback": f"Syntax error: {e}"}
    except Exception as e:
        return {"passed": False, "score": 0, "feedback": f"Error: {e}"}

# Run Reflexion on a coding task
reflexion_agent(
    task="Write a function top_k_smallest(lst, k) returning the k smallest elements sorted. "
         "Edge cases: empty list→[], k=0→[], k>len→full sorted list.",
    evaluator=code_evaluator,
)


💡 **Key takeaway:** Reflexion works because reflections accumulate — on attempt 3, the agent knows what failed in attempts 1 and 2. The evaluator is key: without automated testing, Reflexion is just expensive regeneration. Use it for code, SQL, math — anything testable.


---

## 7. Building an MCP Server

MCP (Model Context Protocol) standardises how agents connect to tools. Write one server — **any** MCP-compatible agent can use it (Claude, ChatGPT, Cursor, your custom agent).


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mcp", "nest_asyncio"])
import nest_asyncio
nest_asyncio.apply()
print("✅ MCP SDK + nest_asyncio installed")


### 7.1 — Write the MCP Server to a File

The server runs as a separate process. We write it to disk, then connect from this notebook.


In [ ]:
server_code = """from mcp.server import Server
from mcp.types import TextContent

server = Server("techstore")

PRODUCTS = {
    "laptop-pro": {"name": "TechStore Laptop Pro", "price": 1299.99, "qty": 23, "category": "laptops"},
    "wireless-mouse": {"name": "ErgoClick Wireless Mouse", "price": 49.99, "qty": 156, "category": "accessories"},
    "usb-hub": {"name": "USB-C Hub 7-in-1", "price": 34.99, "qty": 0, "category": "accessories"},
    "monitor-4k": {"name": "UltraView 4K Monitor 27in", "price": 449.99, "qty": 12, "category": "monitors"},
    "keyboard-mech": {"name": "MechType Pro Keyboard", "price": 129.99, "qty": 67, "category": "accessories"},
}

@server.tool()
async def check_inventory(product_name: str) -> list[TextContent]:
    '''Check product availability and price. Accepts partial names.'''
    matches = [p for p in PRODUCTS.values() if product_name.lower() in p["name"].lower()]
    if not matches:
        return [TextContent(type="text", text=f"No products matching '{product_name}'.")]
    lines = [f"  {p['name']}: ${p['price']:.2f} — {'In Stock' if p['qty']>0 else 'OUT OF STOCK'} ({p['qty']})" for p in matches]
    return [TextContent(type="text", text="\\n".join(lines))]

@server.tool()
async def search_products(query: str = "", category: str = "") -> list[TextContent]:
    '''Search product catalog by name or category.'''
    results = list(PRODUCTS.values())
    if query:
        results = [p for p in results if query.lower() in p["name"].lower()]
    if category:
        results = [p for p in results if p["category"] == category.lower()]
    if not results:
        return [TextContent(type="text", text="No products found.")]
    lines = [f"  {p['name']} — ${p['price']:.2f}" for p in results]
    return [TextContent(type="text", text="\\n".join(lines))]

if __name__ == "__main__":
    import asyncio
    from mcp.server.stdio import stdio_server
    async def main():
        async with stdio_server() as (r, w):
            await server.run(r, w)
    asyncio.run(main())
"""

with open("/tmp/techstore_mcp_server.py", "w") as f:
    f.write(server_code)

print("✅ MCP server written to /tmp/techstore_mcp_server.py")
print("   Tools: check_inventory, search_products")


### 7.2 — Connect to the MCP Server and Call Tools


In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def test_mcp():
    params = StdioServerParameters(command=sys.executable, args=["/tmp/techstore_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print(f"🔌 MCP connected — {len(tools.tools)} tools:")
            for t in tools.tools:
                print(f"   📦 {t.name}: {(t.description or '')[:60]}")

            print(f"\n{'─' * 50}")
            result = await session.call_tool("check_inventory", {"product_name": "laptop"})
            print(f"\n🔧 check_inventory('laptop'):\n{result.content[0].text}")

            result = await session.call_tool("search_products", {"category": "accessories"})
            print(f"\n🔧 search_products(category='accessories'):\n{result.content[0].text}")

asyncio.run(test_mcp())


💡 **Key takeaway:** MCP end-to-end: server written once (~40 lines), any MCP client can discover and call those tools. Claude Desktop, OpenAI Agents SDK, Cursor — zero changes to the server.


---

## 8. Architecture Comparison

Same question, two approaches. Watch the difference in how they arrive at the answer.


In [ ]:
q = "What's Tesla's Q4 2025 revenue, and what's that per month?"

print("▓" * 60)
print("▓  APPROACH 1: ReAct (Basic Agent Loop)")
print("▓" * 60)
t1 = time.time()
r1 = agent(q)
t1 = time.time() - t1

print("\n" + "▓" * 60)
print("▓  APPROACH 2: Plan-and-Execute")
print("▓" * 60)
t2 = time.time()
r2 = plan_and_execute(q)
t2 = time.time() - t2

print("\n" + "═" * 60)
print(f"ReAct:            {t1:.1f}s")
print(f"Plan-and-Execute: {t2:.1f}s")
print("Same answer, different approach — reactive vs deliberate.")


💡 **Key takeaway:** For simple questions (1–3 steps), ReAct is faster — no planning overhead. For complex tasks (4+ steps), Plan-and-Execute often uses fewer total tokens. Choose based on the task.


---

## 9. Practice Challenges

### Challenge 1: Multi-Source Agent
Add `get_stock_price(ticker)` and `convert_currency(amount, from_cur, to_cur)` tools. Ask: *"What's Apple's stock price in UAE dirhams?"* — the agent should chain both tools.

### Challenge 2: Reflexion for SQL
Write an evaluator that runs SQL against a test SQLite database. Use Reflexion to generate a correct query for: *"Top 3 product categories by revenue in Q3 2025."*

### Challenge 3: Extend the MCP Server
Add `create_return_request(order_id, reason)` and `get_recommendations(category, budget)`. Connect to the MCP agent and test: *"Return ORD-10001 (damaged) and recommend a replacement laptop under $1500."*


---

## Wrap-Up

You started with an LLM that could only follow instructions. You're ending with three agent architectures, each making autonomous decisions.

What you built:

1. **The Agent Loop** — `for` loop + `if tool_calls` = the core of every agent
2. **ReAct** — explicit Thought/Action/Observation traces for debuggability
3. **Plan-and-Execute** — separate planner from executor for multi-step tasks
4. **Reflexion** — self-critique with automated evaluation for verifiable output
5. **MCP Server** — tools exposed via standard protocol, any client can use them

The critical insight: **all three architectures use the same core loop.** The differences are *when* the agent plans (upfront vs reactive), *whether* it self-evaluates (Reflexion), and *how* it connects to tools (hardcoded vs MCP).

In **Section 11**, you'll wrap these patterns in production frameworks — LangGraph for state machines, CrewAI for multi-agent teams, and OpenAI Agents SDK for handoff-based systems. Now you know what's under the hood.
